# CrossVul-DG verified results analysis

Only completed protocol-v2 runs are accepted. Each backbone/configuration/data/source cohort is plotted separately. No synthetic or illustrative fallback metrics are supplied. CC-DRO is included in every method chart when results exist.

AP is Average Precision. The F1 below is vulnerable-class binary F1, averaged over projects after averaging seeds. DA methods use unlabelled target data; PD-DRO uses additional labelled tail data. Test PR-curve R@P50 is diagnostic; validation-selected operating points have separate metric fields.

Defaults below show the completed bounded real pilot (one fold/seed, baseline lambda=1). A zero one-observation standard deviation is not a confidence interval or robustness claim. Change RUNS_DIR and expected counts for full-population studies.


In [ ]:
import sys
from pathlib import Path
import matplotlib.pyplot as plt
ROOT = Path.cwd() if (Path.cwd() / "scripts").is_dir() else Path.cwd().parent
sys.path.insert(0, str(ROOT / "scripts"))
from aggregate_results import load_runs, summarize, METHOD_ORDER, METHOD_LABEL
RUNS_DIR = ROOT / "runs" / "real_baselines" / "bounded-v1"
EXPECTED_FOLDS, EXPECTED_SEEDS = 1, 1
bucket, metadata = load_runs(RUNS_DIR)
summary = summarize(bucket, expected_folds=EXPECTED_FOLDS, expected_seeds=EXPECTED_SEEDS)
print(f"Verified cohorts: {len(summary)}")


In [ ]:
for index, (cohort, result) in enumerate(summary.items()):
    methods = [m for m in METHOD_ORDER if m in result["methods"]]
    fig, ax = plt.subplots(figsize=(10, 4))
    means = [100 * result["methods"][m]["pr_auc"]["mean"] for m in methods]
    errors = [100 * result["methods"][m]["pr_auc"]["std"] for m in methods]
    ax.bar(range(len(methods)), means, yerr=errors, capsize=4)
    ax.set_xticks(range(len(methods)), [METHOD_LABEL[m] for m in methods], rotation=25, ha="right")
    ax.set_ylabel("Average Precision (%)")
    ax.set_title(f"{metadata[cohort]['backbone']} | cohort {index} | {result['status']}")
    fig.tight_layout()
    (ROOT / "figures").mkdir(exist_ok=True)
    fig.savefig(ROOT / "figures" / f"cohort_{index}_ap.png", dpi=160)
    plt.show()


## Pseudo-domain cluster ablations

`resolved_config` is the configuration mapping; `args.config` is a filename. Each cluster count remains in its own cohort. Before comparing cluster counts, confirm that source/data/backbone/seed/fold identities match and only `pseudo.n_clusters` differs. The cell below inventories available measurements without averaging incompatible runs.


In [ ]:
cluster_inventory = []
for cohort, info in metadata.items():
    if "pd-dro" in bucket[cohort]:
        cluster_inventory.append({"cohort": cohort, "n_clusters": info["config"]["pseudo"]["n_clusters"],
                                  "backbone": info["backbone"], "folds": sorted(bucket[cohort]["pd-dro"])})
cluster_inventory


Generate cohort-specific tables and `summary.json` with:

```sh
python scripts/aggregate_results.py --runs_dir runs --out paper_tables --expected_folds 10 --expected_seeds 3 --expected_methods erm irm dro cc-dro pd-dro da-coral da-dann
```

Use `--allow_incomplete` only for clearly labelled exploratory tables. It disables paired significance tests for incomplete cohorts.
